# Queues

## Python's Built-in `deque`

In [ ]:

from collections import deque

py_queue = deque()

py_queue.append("task 1")
py_queue.append("task 2")
py_queue.append("task 3")
print(f"Current queue: {py_queue}")

first_item = py_queue.popleft()
print(f"Dequeued: {first_item}")
print(f"Queue after dequeue: {py_queue}")

front_item = py_queue[0]
print(f"Front item: {front_item}")

is_empty = not py_queue
print(f"Is empty: {is_empty}")

`deque` (double-ended queue) is Python's built-in, ready-to-use queue. `append()` adds to the rear, `popleft()` removes from the front — both in O(1) time, which is why `deque` is the go-to choice over a plain list for real queues. Peeking at the front is just indexing (`py_queue[0]`), and since an empty deque is "falsy", `not py_queue` is a quick empty check.

## A Naive Array-Backed Queue

In [ ]:

class ArrayQueue:
    def __init__(self):
        self.queue = []

    def is_empty(self):
        return len(self.queue) == 0

    def enqueue(self, item):
        self.queue.append(item)
        print(f"enqueued {item}")

    def dequeue(self):
        if self.is_empty():
            raise Exception("Elements cannot be removed from an empty queue")
        return self.queue.pop(0)

    def front_element(self):
        if self.is_empty():
            raise Exception("Queue is empty")
        return self.queue[0]

    def size(self):
        return len(self.queue)


a = ArrayQueue()
a.enqueue(1)
a.enqueue(2)
a.enqueue(1)
a.enqueue(2)
a.dequeue()
print(a.is_empty())
print(a.size())
print(a.front_element())

This builds a queue from scratch using a plain Python list. `enqueue` adds to the back with `append()`, which is cheap. But `dequeue` has to remove from the **front** with `pop(0)`, and that's the catch: removing the first element of a list means every remaining element has to shift left by one, making it an O(n) operation. That's the classic downside of backing a queue with a plain array/list instead of something like `deque`.

## A Naive Fixed-Size Queue

In [ ]:

class ListQueue:
    def __init__(self, capacity=3):
        self.items = []
        self.capacity = capacity

    def is_full(self):
        return len(self.items) == self.capacity

    def is_empty(self):
        return len(self.items) == 0

    def enqueue(self, data):
        if self.is_full():
            print("The queue is full")
        else:
            self.items.append(data)

    def dequeue(self):
        if self.is_empty():
            print("Queue is empty")
        else:
            return self.items.pop(0)


q = ListQueue()
q.enqueue(20)
q.enqueue(30)
q.enqueue(40)
q.enqueue(50)
print(q.items)
q.dequeue()
q.dequeue()
print(q.items)

A simpler variant that caps how many items the queue can hold. Once `capacity` items are in, any further `enqueue` calls are rejected instead of growing the list forever. Like `ArrayQueue`, it still pays the O(n) cost on `dequeue()` because of `pop(0)` — the fixed size just adds a ceiling on top.

## Circular Queue (O(1) Enqueue and Dequeue)

In [ ]:

class CircularQueue:
    def __init__(self, capacity):
        self.capacity = capacity
        self.queue = [None] * capacity
        self.front = -1
        self.rear = -1
        self.current_size = 0

    def is_empty(self):
        return self.current_size == 0

    def is_full(self):
        return self.current_size == self.capacity

    def enqueue(self, item):
        if self.is_full():
            raise Exception("Queue Overflow: Cannot enqueue to a full queue.")

        if self.is_empty():
            self.front = 0

        self.rear = (self.rear + 1) % self.capacity
        self.queue[self.rear] = item
        self.current_size += 1

        print(f"Enqueued {item} to the queue.")

    def dequeue(self):
        if self.is_empty():
            raise Exception("Queue Underflow: Cannot dequeue from an empty queue.")

        item = self.queue[self.front]
        self.queue[self.front] = None
        self.current_size -= 1

        if self.is_empty():
            self.front = -1
            self.rear = -1
        else:
            self.front = (self.front + 1) % self.capacity

        return item

    def front_element(self):
        if self.is_empty():
            raise Exception("Queue is empty.")
        return self.queue[self.front]

    def size(self):
        return self.current_size


cq = CircularQueue(3)
cq.enqueue(10)
cq.enqueue(20)
cq.enqueue(30)

print(f"Front element is: {cq.front_element()}")
print(f"Dequeued: {cq.dequeue()}")

cq.enqueue(40)

print(f"Front element is now: {cq.front_element()}")
print(f"Queue: {cq.queue}, Front: {cq.front}, Rear: {cq.rear}")

This is the fix for the O(n) problem above. Instead of shifting elements, `front` and `rear` are just indices that **wrap around** the fixed-size array using `% capacity` — once `rear` reaches the last slot, the next enqueue wraps back to index 0 (as long as there's room). Both `enqueue` and `dequeue` only ever touch one slot and move one pointer, so both are O(1), with a `current_size` counter used to tell full from empty (since both cases can otherwise look identical).

# Stacks

In [ ]:

class Stack:
    def __init__(self, capacity):
        self.capacity = capacity
        self.stack = [None] * capacity
        self.top = -1

    def is_empty(self):
        return self.top == -1

    def is_full(self):
        return self.capacity == self.top + 1

    def size(self):
        return self.top + 1

    def push(self, item):
        if self.is_full():
            raise Exception("Stack Overflow!")
        self.top += 1
        self.stack[self.top] = item

    def pop(self):
        if self.is_empty():
            raise Exception("Stack Underflow!")
        item = self.stack[self.top]
        self.stack[self.top] = None
        self.top -= 1
        return item

    def peek(self):
        if self.is_empty():
            raise Exception("Stack Underflow!")
        return self.stack[self.top]

A stack is LIFO: last in, first out. `top` is just an index into a fixed-size array pointing at the most recently pushed item. `push` moves `top` forward and drops the item in; `pop` reads the item at `top` and moves `top` back; `peek` reads it without moving anything. Think of a stack of plates — you can only add or remove from the top.

## Stack Applications

In [ ]:

def postfix_evaluation(expression):
    st = Stack(len(expression.split()))
    tokens = expression.split()

    for token in tokens:
        if token.isdigit():
            st.push(int(token))
        else:
            val2 = st.pop()
            val1 = st.pop()
            if token == "+":
                st.push(val1 + val2)
            elif token == "-":
                st.push(val1 - val2)
            elif token == "*":
                st.push(val1 * val2)
            elif token == "/":
                st.push(val1 / val2)

    return st.pop()


def valid_parenthesis(brackets):
    st = Stack(len(brackets))
    match = {")": "(", "}": "{", "]": "["}

    for char in brackets:
        if char in match.values():
            st.push(char)
        elif char in match:
            if st.is_empty():
                return False
            top_element = st.pop()
            if top_element != match[char]:
                return False

    return st.is_empty()


def infix_to_postfix(expression):
    precedence = {"+": 1, "-": 1, "*": 2, "/": 2}
    st = Stack(len(expression))
    output = []
    tokens = expression.split()

    for token in tokens:
        if token.isalnum():
            output.append(token)
        elif token == "(":
            st.push(token)
        elif token == ")":
            while st.peek() != "(":
                output.append(st.pop())
            st.pop()
        elif token in precedence:
            while (not st.is_empty() and st.peek() != "(" and
                   precedence.get(st.peek(), 0) >= precedence[token]):
                output.append(st.pop())
            st.push(token)

    while not st.is_empty():
        output.append(st.pop())

    return " ".join(output)


print(postfix_evaluation("3 4 +"))
print(valid_parenthesis("{[()]}"))
print(valid_parenthesis("{[(])}"))
print(infix_to_postfix("3 + 4 * 2"))

Three classic stack tricks, each leaning on the "last in, first out" behavior:

- **Postfix evaluation** — in postfix notation (`3 4 +` instead of `3 + 4`), operators come *after* their operands. Numbers get pushed onto the stack as they're read; when an operator shows up, the two most recent numbers are popped, combined, and the result is pushed back. Whatever's left on the stack at the end is the answer.
- **Valid parentheses** — every opening bracket gets pushed. When a closing bracket appears, it should match whatever's currently on top of the stack; if it doesn't (or the stack is empty when it shouldn't be), the brackets aren't balanced.
- **Infix to postfix** — converts normal notation (`3 + 4 * 2`) into postfix, using the stack to temporarily hold operators until it's the right moment (based on precedence) to move them into the output.

These were turned into standalone functions (rather than living inside `Stack` itself) since each one just needs its own fresh, temporary stack to work with — they aren't really operations *of* a stack, they're algorithms that *use* one.